# Random forest

Random forest with 100 trees on the 3328 per-antenna-normalised amplitudes of each frame, evaluated on the 80 cluster-held-out splits. Results are written to `results/`.

## Step 1: Settings

100 trees with the Gini criterion and scikit-learn defaults otherwise.

In [ ]:
import os
import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
                             confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve)

DATA_DIR = "../output_raw_full"
RESULTS_DIR = "results"
SVM_RESULTS_DIR = "../SVM/results"

N_ESTIMATORS = 100
CRITERION = "gini"
MAX_FEATURES = "sqrt"
SEED = 42
N_JOBS = -1

CLASS_NAMES = ["NLOS (0)", "LOS (1)"]

COLOR_LOS = "#2a78d6"
COLOR_NLOS = "#eb6834"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e4e3df"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": INK_2, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "lines.linewidth": 2,
})

os.makedirs(RESULTS_DIR, exist_ok=True)

## Step 2: Splits

`lopo` holds out one LOS and one NLOS cluster (20 splits). `l2po` is the L2PO protocol and holds out two LOS and two NLOS clusters (60 splits). `TRAIN_SUBSAMPLE = None` uses all training frames.

In [ ]:
import itertools

LOS_CLUSTERS = ["LOS_cluster1", "LOS_cluster2", "LOS_cluster3", "LOS_cluster4"]
NLOS_CLUSTERS = ["NLOS_cluster1", "NLOS_cluster2", "NLOS_cluster3", "NLOS_cluster4", "NLOS_cluster5"]
ALL_CLUSTERS = LOS_CLUSTERS + NLOS_CLUSTERS

PROTOCOLS = ["lopo", "l2po"]
TRAIN_SUBSAMPLE = None

short = lambda c: c.replace("_cluster", "")

def make_splits(protocol):
    if protocol == "lopo":
        tests = [[l, n] for l in LOS_CLUSTERS for n in NLOS_CLUSTERS]
    elif protocol == "l2po":
        tests = [list(ls) + list(ns) for ls in itertools.combinations(LOS_CLUSTERS, 2)
                 for ns in itertools.combinations(NLOS_CLUSTERS, 2)]
    else:
        raise ValueError(protocol)
    splits = []
    for test in tests:
        train = [c for c in ALL_CLUSTERS if c not in test]
        splits.append({"name": "test=" + "+".join(short(c) for c in test), "train": train, "test": test})
    return splits

SPLITS = {p: make_splits(p) for p in PROTOCOLS}
for p, splits in SPLITS.items():
    print(f"{p}: {len(splits)} splits")
    for i, sp in enumerate(splits, 1):
        print(f"  {i:2d}. train={'+'.join(short(c) for c in sp['train']):38s} {sp['name']}")

## Step 3: Load the per-cluster features

In [ ]:
CLUSTER_DIR = os.path.join(DATA_DIR, "clusters")
with open(os.path.join(CLUSTER_DIR, "index.json")) as f:
    cidx = json.load(f)
assert cidx["norm"] == "antenna" and not cidx["zscore_applied"], "re-run preprocess_raw.ipynb with NORM = 'antenna'"

cluster_X = {c: np.load(os.path.join(CLUSTER_DIR, cidx["clusters"][c]["features_file"])) for c in ALL_CLUSTERS}
cluster_y = {c: cidx["clusters"][c]["label"] for c in ALL_CLUSTERS}
for c in ALL_CLUSTERS:
    print(f"{c:14s} label={cluster_y[c]}  X={cluster_X[c].shape}")

## Step 4: Evaluation of one split

Trees are invariant to feature scaling, so no z-score is applied.

In [ ]:
from sklearn.metrics import f1_score, recall_score

def evaluate_split(split, seed=SEED):
    rng = np.random.default_rng(seed)
    Xtr = np.concatenate([cluster_X[c] for c in split["train"]])
    ytr = np.concatenate([np.full(len(cluster_X[c]), cluster_y[c], dtype=np.int8) for c in split["train"]])
    if TRAIN_SUBSAMPLE and TRAIN_SUBSAMPLE < len(ytr):
        keep = np.sort(rng.choice(len(ytr), TRAIN_SUBSAMPLE, replace=False))
        Xtr, ytr = Xtr[keep], ytr[keep]

    model = RandomForestClassifier(n_estimators=N_ESTIMATORS, criterion=CRITERION, max_features=MAX_FEATURES,
                                   n_jobs=N_JOBS, random_state=seed)
    t0 = time.time()
    model.fit(Xtr, ytr)
    fit_s = time.time() - t0
    del Xtr

    y_true, y_hat, score, per_cluster_acc = [], [], [], {}
    for c in split["test"]:
        p = model.predict_proba(cluster_X[c])[:, 1]
        pred = (p >= 0.5).astype(np.int8)
        yt = np.full(len(pred), cluster_y[c], dtype=np.int8)
        per_cluster_acc[c] = float((pred == yt).mean())
        y_true.append(yt); y_hat.append(pred); score.append(p)
    y_true, y_hat, score = np.concatenate(y_true), np.concatenate(y_hat), np.concatenate(score)

    return {
        "name": split["name"], "train": split["train"], "test": split["test"],
        "n_train": int(len(ytr)), "n_test": int(len(y_true)),
        "accuracy": float((y_hat == y_true).mean()),
        "macro_f1": float(f1_score(y_true, y_hat, average="macro")),
        "roc_auc": float(roc_auc_score(y_true, score)),
        "recall_los": float(recall_score(y_true, y_hat, pos_label=1)),
        "recall_nlos": float(recall_score(y_true, y_hat, pos_label=0)),
        "per_cluster_accuracy": per_cluster_acc,
        "fit_seconds": fit_s,
    }

## Step 5: Run all splits

Each split is appended to `results/splits_<protocol>.jsonl`, and finished splits are skipped when the cell runs again. Delete these files after changing a setting.

In [ ]:
def load_done(path):
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {r["name"]: r for r in (json.loads(line) for line in f if line.strip())}

ALL_RESULTS = {}
for protocol, splits in SPLITS.items():
    path = os.path.join(RESULTS_DIR, f"splits_{protocol}.jsonl")
    done = load_done(path)
    print(f"=== {protocol}: {len(splits)} splits ({len(done)} already done)")
    t_start = time.time()
    for i, sp in enumerate(splits, 1):
        if sp["name"] in done:
            continue
        r = evaluate_split(sp)
        with open(path, "a") as f:
            f.write(json.dumps(r) + "\n")
        done[r["name"]] = r
        print(f"  [{i:2d}/{len(splits)}] {r['name']:32s} acc={r['accuracy']:.4f}  macro-F1={r['macro_f1']:.4f}  "
              f"AUC={r['roc_auc']:.4f}  ({r['fit_seconds']:.0f}s fit, {time.time() - t_start:.0f}s total)")
    ALL_RESULTS[protocol] = [done[sp["name"]] for sp in splits]

## Step 6: Summary

In [ ]:
SUMMARY = {}
for protocol, res in ALL_RESULTS.items():
    acc_s = np.array([r["accuracy"] for r in res])
    f1_s = np.array([r["macro_f1"] for r in res])
    auc_s = np.array([r["roc_auc"] for r in res])
    worst = res[int(acc_s.argmin())]
    print(f"=== {protocol} ({len(res)} splits)")
    print(f"  accuracy : {acc_s.mean():.4f} ± {acc_s.std():.4f}   min {acc_s.min():.4f}   max {acc_s.max():.4f}")
    print(f"  macro-F1 : {f1_s.mean():.4f} ± {f1_s.std():.4f}")
    print(f"  ROC-AUC  : {auc_s.mean():.4f} ± {auc_s.std():.4f}")
    print(f"  worst split: {worst['name']}  (acc {worst['accuracy']:.4f})")

    held = {c: [r["per_cluster_accuracy"][c] for r in res if c in r["per_cluster_accuracy"]] for c in ALL_CLUSTERS}
    held = {c: v for c, v in held.items() if v}
    print("  accuracy of each cluster when held out (mean / worst / number of splits):")
    for c in held:
        print(f"    {c:14s} {np.mean(held[c]):.4f} / {np.min(held[c]):.4f} / {len(held[c])}")

    entry = {"n_splits": len(res),
             "accuracy_mean": float(acc_s.mean()), "accuracy_std": float(acc_s.std()),
             "accuracy_min": float(acc_s.min()), "accuracy_max": float(acc_s.max()),
             "macro_f1_mean": float(f1_s.mean()), "macro_f1_std": float(f1_s.std()),
             "roc_auc_mean": float(auc_s.mean()), "roc_auc_std": float(auc_s.std()),
             "worst_split": worst["name"],
             "held_out_cluster_accuracy": {c: {"mean": float(np.mean(v)), "min": float(np.min(v)), "n": len(v)}
                                           for c, v in held.items()}}
    SUMMARY[protocol] = entry
    print()

## Step 7: Figures and saved summary

In [ ]:
for protocol, res in ALL_RESULTS.items():
    acc_s = np.array([r["accuracy"] for r in res])
    order = np.argsort(acc_s)

    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(np.arange(len(acc_s)), acc_s[order], "o", color=COLOR_LOS, markersize=6)
    ax.axhline(acc_s.mean(), color=INK_2, linestyle="--", linewidth=1)
    ax.text(0, acc_s.mean(), f" mean {acc_s.mean():.3f}", va="bottom", color=INK_2)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_xlabel("Split (sorted by accuracy)")
    ax.set_ylabel("Test accuracy")
    ax.set_title(f"RF: accuracy of all {len(acc_s)} {protocol} splits")
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"splits_{protocol}_accuracy.png"), dpi=150)
    plt.show()

    held = SUMMARY[protocol]["held_out_cluster_accuracy"]
    shown = [c for c in ALL_CLUSTERS if c in held]
    means = np.array([held[c]["mean"] for c in shown])
    mins = np.array([held[c]["min"] for c in shown])
    maxs = np.array([max(r["per_cluster_accuracy"][c] for r in res if c in r["per_cluster_accuracy"]) for c in shown])
    colors = [COLOR_LOS if c.startswith("LOS") else COLOR_NLOS for c in shown]

    fig, ax = plt.subplots(figsize=(8, 4.6))
    y = np.arange(len(shown))
    ax.set_axisbelow(True)
    ax.grid(axis="y", visible=False)
    ax.barh(y, means, color=colors, height=0.6, edgecolor="#fcfcfb", linewidth=2)
    ax.errorbar(means, y, xerr=[means - mins, maxs - means], fmt="none", ecolor=INK_2, capsize=3, linewidth=1)
    for yi, m, hi in zip(y, means, maxs):
        ax.text(hi + 0.015, yi, f"{m:.3f}", va="center", color=INK, fontsize=9)
    ax.set_yticks(y, [short(c) for c in shown])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("Accuracy when the cluster is held out (mean; bar = worst to best split)")
    ax.set_title(f"RF ({protocol}): accuracy per held-out cluster")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLOR_LOS), plt.Rectangle((0, 0), 1, 1, color=COLOR_NLOS)],
              labels=["LOS cluster", "NLOS cluster"], frameon=False, ncol=2,
              loc="upper center", bbox_to_anchor=(0.5, -0.16))
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"splits_{protocol}_per_cluster.png"), dpi=150)
    plt.show()

with open(os.path.join(RESULTS_DIR, "splits_summary.json"), "w") as f:
    json.dump({"settings": {"n_estimators": N_ESTIMATORS, "criterion": CRITERION, "max_features": MAX_FEATURES,
                            "train_subsample": TRAIN_SUBSAMPLE, "seed": SEED},
               "protocols": SUMMARY}, f, indent=2)
print("saved:", os.path.join(RESULTS_DIR, "splits_summary.json"))

## Step 8: Comparison with the SVM on the same splits

In [ ]:
for protocol, res in ALL_RESULTS.items():
    svm_path = os.path.join(SVM_RESULTS_DIR, f"splits_{protocol}.jsonl")
    if not os.path.exists(svm_path):
        print(f"{protocol}: SVM results not found ({svm_path}) -- skipped")
        continue
    svm_res = load_done(svm_path)
    common = [r for r in res if r["name"] in svm_res]
    rf_acc = np.array([r["accuracy"] for r in common])
    svm_acc = np.array([svm_res[r["name"]]["accuracy"] for r in common])
    diff = rf_acc - svm_acc

    print(f"=== {protocol}: {len(common)} common splits")
    print(f"  SVM accuracy : {svm_acc.mean():.4f} ± {svm_acc.std():.4f}   (min {svm_acc.min():.4f})")
    print(f"  RF  accuracy : {rf_acc.mean():.4f} ± {rf_acc.std():.4f}   (min {rf_acc.min():.4f})")
    print(f"  RF - SVM     : {diff.mean():+.4f} on average;  RF better on {(diff > 0).sum()}, "
          f"SVM better on {(diff < 0).sum()}, equal on {(diff == 0).sum()} splits")

    print(f"  {'held-out cluster':16s} {'SVM':>8s} {'RF':>8s}")
    for c in ALL_CLUSTERS:
        s_v = [svm_res[r["name"]]["per_cluster_accuracy"][c] for r in common if c in r["per_cluster_accuracy"]]
        r_v = [r["per_cluster_accuracy"][c] for r in common if c in r["per_cluster_accuracy"]]
        if r_v:
            print(f"  {c:16s} {np.mean(s_v):8.4f} {np.mean(r_v):8.4f}")
    print()